<a href="https://colab.research.google.com/github/team0243/map_test/blob/main/Mapmu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install folium #ติดตั้ง แพคเกจ folium

In [ ]:
!pip install --upgrade -q gspread pandas

In [ ]:
import pandas as pd    # pandas ใช้จัดการข้อมูลที่เป็นตาราง
import os # os ใช้จัดการกับไฟล์ ว่าอยู่ที่ไหน
import folium #เรียกใช้แพคเกจ
from folium import plugins #
from folium.plugins import MarkerCluster #ติดตั้ง ปลั๊กอิน MakerCluster

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
from folium.plugins import Search, LocateControl

In [ ]:
# Replace with your Google Sheets URL
url = 'https://docs.google.com/spreadsheets/d/1dKSYqN1ud_QF2vmvWWEfjFR8v0OAjzl1tYzdCiEIGwg/edit?gid=0#gid=0'
# Replace with your specific sheet tab name, e.g., 'Sheet1' (optional)
sheet_name = 'Sheet1'

sh = gc.open_by_url(url)

if sheet_name:
    worksheet = sh.worksheet(sheet_name)
else:
    # Use the first sheet if no name is specified
    worksheet = sh.get_worksheet(0)

# Get all values as a list of lists
rows = worksheet.get_all_values()


In [ ]:
import pandas as pd

# Assuming 'rows' is already populated from the previous steps
# If 'rows' is not defined, this cell will throw an error.
# To be robust, one might check if 'rows' is defined or re-run the sheet loading code.

# Convert the list of lists to a pandas DataFrame for better display
if 'rows' in globals() and rows:
    # Assuming the first row contains headers
    df = pd.DataFrame(rows[1:], columns=rows[0])
    print("First 5 rows of the data:")
    print(df.head())
else:
    print("No data found in 'rows' variable. Please ensure the Google Sheet was read successfully.")


In [ ]:
df.head()

In [ ]:
import folium
from folium.plugins import MarkerCluster, Search, LocateControl

# -------------------------------
# 1️⃣ สร้าง Map
# -------------------------------
m = folium.Map(
    location=[16.442786, 102.826613],
    zoom_start=13,
    tiles='CartoDB positron'
)

# -------------------------------
# 2️⃣ CSS Popup
# -------------------------------
custom_css = """
<style>
.leaflet-popup-content-wrapper {
    border-radius: 14px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.25);
}
.leaflet-popup-tip {
    background: white;
}
</style>
"""
m.get_root().html.add_child(folium.Element(custom_css))

# -------------------------------
# 3️⃣ ICON
# -------------------------------
icon_dict = {
    "โชคลาภ": "https://raw.githubusercontent.com/team0243/map_test/refs/heads/main/pin.png",
    "ความรัก": "https://raw.githubusercontent.com/team0243/map_test/refs/heads/main/pin.png",
    "การงาน": "https://raw.githubusercontent.com/team0243/map_test/refs/heads/main/pin.png",
    "บุญบารมี": "https://raw.githubusercontent.com/team0243/map_test/refs/heads/main/pin.png",
    "พญานาค": "https://raw.githubusercontent.com/team0243/map_test/refs/heads/main/pin.png"
}

def get_icon(luck_type):
    return icon_dict.get(luck_type, icon_dict["บุญบารมี"])

# -------------------------------
# 4️⃣ Layer Filter
# -------------------------------
layers = {
    "โชคลาภ 💰": MarkerCluster(name="โชคลาภ 💰").add_to(m),
    "ความรัก ❤️": MarkerCluster(name="ความรัก ❤️").add_to(m),
    "การงาน 💼": MarkerCluster(name="การงาน 💼").add_to(m),
    "บุญบารมี 🙏": MarkerCluster(name="บุญบารมี 🙏").add_to(m),
    "พญานาค 🐉": MarkerCluster(name="พญานาค 🐉").add_to(m),
}

# -------------------------------
# 5️⃣ Loop Marker (ตัวจริง)
# -------------------------------
for index, row in df.iterrows():

    icon_url = get_icon(row['luck_type'])

    if row['province'] == "ขอนแก่น":
        icon_size = (50, 50)
        icon_anchor = (25, 50)
    else:
        icon_size = (35, 35)
        icon_anchor = (17, 35)

    popup_html = f"""<div style="width:260px;font-family:Prompt,Arial">
        <img src="{row['image']}"
             style="width:100%;height:150px;object-fit:cover;border-radius:10px">
        <div style="margin-top:6px;color:#777;font-size:13px">{row['province']}</div>
        <div style="font-size:18px;font-weight:bold;color:#222">{row['name']}</div>
        <div style="margin:6px 0">
            <span style="background:#ffe9b3;padding:3px 8px;border-radius:6px;font-size:12px;color:#9c6b00">
                เด่นเรื่อง {row['luck_type']}
            </span>
            <span style="background:#e8f2ff;padding:3px 8px;border-radius:6px;font-size:12px;color:#0057b8">
                {row['type']}
            </span>
        </div>
        <div style="font-size:14px;color:#444">{row['description']}</div>

        <div style="margin-top:8px;display:flex;gap:6px">
            <a href="{row['web_project_path']}" target="_blank"
               style="text-decoration:none;background:#ff6a00;color:white;padding:5px 8px;border-radius:6px;font-size:13px">
               ดูเพิ่มเติม
            </a>

            <a href="https://www.google.com/maps?q={row['lat']},{row['lng']}"
               target="_blank"
               style="text-decoration:none;background:#1d4ed8;color:white;padding:5px 8px;border-radius:6px;font-size:13px">
               นำทาง
            </a>
        </div>
    </div>"""

    marker = folium.Marker(
        location=[row['lat'], row['lng']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=row['name'],
        icon=folium.features.CustomIcon(
            icon_url,
            icon_size=icon_size,
            icon_anchor=icon_anchor
        )
    )

    # ใส่ Layer
    if row['luck_type'] == "โชคลาภ":
        marker.add_to(layers["โชคลาภ 💰"])
    elif row['luck_type'] == "ความรัก":
        marker.add_to(layers["ความรัก ❤️"])
    elif row['luck_type'] == "การงาน":
        marker.add_to(layers["การงาน 💼"])
    elif row['luck_type'] == "บุญบารมี":
        marker.add_to(layers["บุญบารมี 🙏"])
    elif row['luck_type'] == "พญานาค":
        marker.add_to(layers["พญานาค 🐉"])
    else:
        marker.add_to(layers["บุญบารมี 🙏"])

# -------------------------------
# 6️⃣ GeoJson Search (ซ่อน marker)
# -------------------------------
features = []

for index, row in df.iterrows():
    features.append({
        "type": "Feature",
        "properties": {
            "name": str(row["name"]),
        },
        "geometry": {
            "type": "Point",
            "coordinates": [row["lng"], row["lat"]],
        },
    })

geojson = folium.GeoJson(
    {
        "type": "FeatureCollection",
        "features": features
    },
    name="SearchLayer",
    marker=folium.CircleMarker(
        radius=0,
        opacity=0,
        fill_opacity=0
    )
).add_to(m)

# -------------------------------
# 7️⃣ Search
# -------------------------------
Search(
    layer=geojson,
    search_label='name',
    placeholder='🔍 ค้นหาวัด...',
    collapsed=False,
    position='topleft'
).add_to(m)

# -------------------------------
# 8️⃣ GPS
# -------------------------------
LocateControl(position='topleft').add_to(m)

# -------------------------------
# 9️⃣ Layer Control (ต้องสุดท้าย)
# -------------------------------
folium.LayerControl(position='topright', collapsed=False).add_to(m)

# -------------------------------
# 🔟 Legend
# -------------------------------
legend_html = """
<div style="
position: fixed;
bottom: 40px; left: 40px;
width: 160px;
background: white;
padding: 10px;
border-radius: 10px;
box-shadow: 0 0 10px rgba(0,0,0,0.2);
font-size:14px;
z-index:9999;
">
<b>สายมู</b><br><br>
💰 โชคลาภ<br>
❤️ ความรัก<br>
💼 การงาน<br>
🙏 บุญบารมี<br>
🐉 พญานาค
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

In [ ]:
m

In [ ]:
# Save the Folium map to an HTML file
map_filename = 'folium_map.html'
m.save(map_filename)
print(f"Map saved to {map_filename}")

# You can then download this file from the Colab file browser (left sidebar -> folder icon)